In [1]:
# ======================================================
# 1. Install required packages
# ======================================================
!pip install ucimlrepo kagglehub --quiet


In [2]:
# ======================================================
# 2. Import libraries
# ======================================================
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from ucimlrepo import fetch_ucirepo
import tensorflow as tf
import os


In [9]:
# ======================================================
# Function to compute cumulative variance explained (PCA)
# ======================================================
def pca_variance_explained(X, retention_levels):
    """
    Compute cumulative variance explained for PCA
    at given retention levels.
    """
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    original_dims = X_scaled.shape[1]

    # Fit full PCA once
    pca_full = PCA(n_components=original_dims)
    pca_full.fit(X_scaled)

    cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

    results = []

    for r in retention_levels:
        n_components = max(1, int(original_dims * r))
        var_explained = cumulative_variance[n_components - 1] * 100

        results.append({
            'Retention Level (%)': int(r * 100),
            'PCA Components': n_components,
            'Variance Explained (%)': var_explained
        })

    return pd.DataFrame(results)


In [4]:
# ======================================================
# 3. Define retention levels to test
# ======================================================
retention_levels = [0.10, 0.25, 0.50, 0.75, 0.90]


In [5]:
# ======================================================
# 4. Load Wine dataset
# ======================================================
wine_data = fetch_ucirepo(id=186)  # Wine Quality dataset
X_wine = wine_data.data.features

# Compute PCA and MSE
wine_results = pca_reconstruction_mse(X_wine, retention_levels)
wine_results['Dataset'] = 'Wine'

wine_results


,Retention Level (%),PCA Components,Reconstruction MSE,Dataset
0,10,1,0.724557,Wine
1,25,2,0.497846,Wine
2,50,5,0.202685,Wine
3,75,8,0.054323,Wine
4,90,9,0.023684,Wine


In [10]:
wine_variance = pca_variance_explained(X_wine, retention_levels)
wine_variance['Dataset'] = 'Wine'
wine_variance


,Retention Level (%),PCA Components,Variance Explained (%),Dataset
0,10,1,27.544260,Wine
1,25,2,50.215406,Wine
2,50,5,79.731533,Wine
3,75,8,94.567722,Wine
4,90,9,97.631577,Wine


In [6]:
# ======================================================
# 5. Load Breast Cancer dataset
# ======================================================
breast_data = fetch_ucirepo(id=17)  # Breast Cancer Wisconsin Diagnostic
X_breast = breast_data.data.features

# Compute PCA and MSE
breast_results = pca_reconstruction_mse(X_breast, retention_levels)
breast_results['Dataset'] = 'Breast Cancer'

breast_results


,Retention Level (%),PCA Components,Reconstruction MSE,Dataset
0,10,3,0.273636,Breast Cancer
1,25,7,0.089905,Breast Cancer
2,50,15,0.013512,Breast Cancer
3,75,22,0.002514,Breast Cancer
4,90,27,0.000082,Breast Cancer


In [11]:
breast_variance = pca_variance_explained(X_breast, retention_levels)
breast_variance['Dataset'] = 'Breast Cancer'
breast_variance


,Retention Level (%),PCA Components,Variance Explained (%),Dataset
0,10,3,72.636371,Breast Cancer
1,25,7,91.009530,Breast Cancer
2,50,15,98.648812,Breast Cancer
3,75,22,99.748579,Breast Cancer
4,90,27,99.991763,Breast Cancer


In [7]:
# ======================================================
# 6. Load MNIST dataset
# ======================================================
(X_train, _), (_, _) = tf.keras.datasets.mnist.load_data()

# Flatten images for PCA
X_mnist = X_train.reshape(X_train.shape[0], -1)  # shape: (60000, 784)

# Compute PCA and MSE
mnist_results = pca_reconstruction_mse(X_mnist, retention_levels)
mnist_results['Dataset'] = 'MNIST'

mnist_results


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


,Retention Level (%),PCA Components,Reconstruction MSE,Dataset
0,10,78,0.322978,MNIST
1,25,196,0.125181,MNIST
2,50,392,0.029531,MNIST
3,75,588,0.005642,MNIST
4,90,705,0.000136,MNIST


In [12]:
mnist_variance = pca_variance_explained(X_mnist, retention_levels)
mnist_variance['Dataset'] = 'MNIST'
mnist_variance


,Retention Level (%),PCA Components,Variance Explained (%),Dataset
0,10,78,64.684139,MNIST
1,25,196,86.312099,MNIST
2,50,392,96.770917,MNIST
3,75,588,99.383085,MNIST
4,90,705,99.985181,MNIST


In [8]:
# ======================================================
# 7. Combine all results into a single table
# ======================================================
comparison_df = pd.concat([wine_results, breast_results, mnist_results], ignore_index=True)

# Reorder columns for readability
comparison_df = comparison_df[['Dataset', 'Retention Level (%)', 'PCA Components', 'Reconstruction MSE']]

# Display the final comparison table
comparison_df


,Dataset,Retention Level (%),PCA Components,Reconstruction MSE
0,Wine,10,1,0.724557
1,Wine,25,2,0.497846
2,Wine,50,5,0.202685
3,Wine,75,8,0.054323
4,Wine,90,9,0.023684
5,Breast Cancer,10,3,0.273636
6,Breast Cancer,25,7,0.089905
7,Breast Cancer,50,15,0.013512
8,Breast Cancer,75,22,0.002514
9,Breast Cancer,90,27,0.000082


In [13]:
# ======================================================
# Table 2: Cumulative Variance Explained (PCA)
# ======================================================
variance_table = pd.concat(
    [wine_variance, breast_variance, mnist_variance],
    ignore_index=True
)

variance_table = variance_table[
    ['Dataset', 'Retention Level (%)', 'PCA Components', 'Variance Explained (%)']
]

variance_table


,Dataset,Retention Level (%),PCA Components,Variance Explained (%)
0,Wine,10,1,27.544260
1,Wine,25,2,50.215406
2,Wine,50,5,79.731533
3,Wine,75,8,94.567722
4,Wine,90,9,97.631577
5,Breast Cancer,10,3,72.636371
6,Breast Cancer,25,7,91.009530
7,Breast Cancer,50,15,98.648812
8,Breast Cancer,75,22,99.748579
9,Breast Cancer,90,27,99.991763
